# 03 · Atribute pe parcelă
Calculează în Earth Engine statisticile pe fiecare parcelă (fără descărcări mari), apoi adaugă ținta din HR-VPP și punctele LUCAS din zonă.

**Regula de bază a modelului:** factorii (pantă, sol, irigare, vânt, cultură) explică; indicii din vara 2024 sunt ținta sau servesc la detecția irigării, **nu** intră ca factori în modelul de vulnerabilitate (ar fi circular).

In [1]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul gisday2026): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/gisday2026.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/gisday2026'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/gisday2026'], check=True)
        os.chdir('/content/gisday2026')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

Lucrez în C:\Users\lefpa\Downloads\gisday2026\gisday2026


In [2]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

Earth Engine OK


In [3]:
S2_BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def s2(start, end):
    def mask(img):
        scl = img.select('SCL')
        bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
        return img.updateMask(bad.Not()).select(S2_BANDS).divide(10000)
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI).filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)).map(mask))

S1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(AOI)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .select(['VV', 'VH']))

In [4]:
import geopandas as gpd, pandas as pd, numpy as np, time
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
g = parcels[['parcel_id', 'geometry']].copy()
g['geometry'] = g.geometry.simplify(5)
g = g.to_crs(4326)
print(len(g), 'parcele')

17466 parcele


In [5]:
bands = []
for m in range(4, 11):
    st = ee.Date.fromYMD(YEAR, m, 1)
    comp = s2(st, st.advance(1, 'month')).median()
    bands += [comp.normalizedDifference(['B8', 'B4']).rename(f'ndvi_{m:02d}'),
              comp.normalizedDifference(['B8', 'B11']).rename(f'ndmi_{m:02d}')]
    s1m = S1.filterDate(st, st.advance(1, 'month')).mean()
    bands += [s1m.select('VV').rename(f'vv_{m:02d}'), s1m.select('VH').rename(f'vh_{m:02d}')]
for y in BASE_YEARS + [YEAR]:
    bands.append(s2(ee.Date.fromYMD(y, 7, 1), ee.Date.fromYMD(y, 9, 1)).median()
                 .normalizedDifference(['B8', 'B4']).rename(f'ndvi_ja_{y}'))
# NDVI din aprilie pe anii de referință: arată ce tip de cultură a fost pe parcelă în fiecare an (ținta din 03e)
for y in BASE_YEARS:
    st = ee.Date.fromYMD(y, 4, 1)
    bands.append(s2(st, st.advance(1, 'month')).median().normalizedDifference(['B8', 'B4']).rename(f'ndvi_apr_{y}'))
# Ploaia lunară 2024 (mm), pentru clasificatorul de irigare (03b)
ERA5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').select('total_precipitation_sum')
for m in range(4, 11):
    st = ee.Date.fromYMD(YEAR, m, 1)
    bands.append(ERA5.filterDate(st, st.advance(1, 'month')).sum().multiply(1000).rename(f'rain_{m:02d}'))
DEM_IC = ee.ImageCollection('COPERNICUS/DEM/GLO30').select('DEM')
dem = DEM_IC.mosaic().setDefaultProjection(DEM_IC.first().projection())
bands += [dem.rename('elev'), ee.Terrain.slope(dem).rename('slope')]
for layer in ('soc', 'sand', 'clay'):
    img = ee.Image(f'projects/soilgrids-isric/{layer}_mean')
    print(layer, img.bandNames().getInfo()[:2])          # verifică benzile reale
    bands.append(img.select(0).rename(f'{layer}_sg_top'))
STACK = ee.Image.cat(bands)
CROP = ee.ImageCollection('JRC/D5/EUCROPMAP/V1').filterDate('2022-01-01', '2023-01-01').first().select(0).rename('crop')

C:\Users\lefpa\Downloads\gisday2026\gisday2026\.venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/DEM/GLO30! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/DEM/GLO30_2024_1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30

  warnings.warn(warning, category=DeprecationWarning)


soc ['soc_0-5cm_mean', 'soc_5-15cm_mean']


sand ['sand_0-5cm_mean', 'sand_5-15cm_mean']


clay ['clay_0-5cm_mean', 'clay_5-15cm_mean']


In [6]:
from gisday_lib import fingerprint

def run_batch(sub, image, reducer, scale):
    fc = ee.FeatureCollection(sub.__geo_interface__)
    res = image.reduceRegions(collection=fc, reducer=reducer, scale=scale, tileScale=4)
    return [f['properties'] for f in res.getInfo()['features']]

# Grupare spațială pe blocuri de 5 km și loturi de 900 (15 apeluri compacte în loc de 90 împrăștiate -> ~6x mai puține EECU)
c_m = parcels.geometry.centroid
order = ((c_m.x // 5000).astype(int) * 1000 + (c_m.y // 5000).astype(int)).argsort(kind='stable')
g_ord = g.iloc[order].reset_index(drop=True)
# Cache-ul depinde de parcele și de benzi: dacă oricare se schimbă, se calculează din nou
BDIR = INTERIM / f'gee_batches_{fingerprint(g_ord, STACK.bandNames().getInfo())}'
BDIR.mkdir(parents=True, exist_ok=True)
N = 900
parts = []
for k in range(0, len(g_ord), N):
    bp = BDIR / f'batch_{k:05d}_{N}.parquet'
    if bp.exists():
        parts.append(pd.read_parquet(bp))
        continue
    sub = g_ord.iloc[k:k + N]
    for attempt in range(3):
        try:
            r1 = run_batch(sub, STACK, ee.Reducer.mean(), 10)
            r2 = run_batch(sub, CROP, ee.Reducer.mode(), 10)
            df_b = pd.DataFrame(r1).merge(pd.DataFrame(r2)[['parcel_id', 'mode']].rename(columns={'mode': 'crop'}),
                                          on='parcel_id', how='left')
            df_b.to_parquet(bp)
            parts.append(df_b)
            break
        except Exception as e:
            print('reîncerc', k, e); time.sleep(10)
    else:
        raise RuntimeError(f'Lotul {k} a eșuat de 3 ori; rulează din nou celula (loturile gata rămân în cache)')
    print(f'{min(k + N, len(g_ord))}/{len(g_ord)}', flush=True)
F = pd.concat(parts, ignore_index=True).sort_values('parcel_id').reset_index(drop=True)
assert len(F) == len(parcels) and F.parcel_id.is_unique, f'{len(F)} rânduri pentru {len(parcels)} parcele'
F.to_parquet(INTERIM / 'parcel_gee_stats.parquet')
F.describe().T.head(20)

900/17466


1800/17466


2700/17466


3600/17466


4500/17466


5400/17466


... (11 outputs left out)

16200/17466


17100/17466


17466/17466


,count,mean,std,min,25%,50%,75%,max
clay_sg_top,17436.0,375.611165,52.713759,214.237801,338.329902,372.709404,415.838642,508.000000
elev,17466.0,71.086416,31.329915,21.003357,43.237228,71.508188,99.911371,122.616119
ndmi_04,17466.0,0.064723,0.267045,-0.370278,-0.166812,-0.026093,0.326164,0.655683
ndmi_05,17466.0,0.148717,0.191238,-0.324945,-0.012906,0.146912,0.294610,0.683756
ndmi_06,17466.0,0.050033,0.149777,-0.319040,-0.066048,0.024529,0.150267,0.684625
ndmi_07,17466.0,-0.088785,0.151739,-0.363734,-0.187396,-0.138836,-0.037475,0.547878
ndmi_08,17466.0,-0.147728,0.145000,-0.383497,-0.243667,-0.192151,-0.107238,0.492664
ndmi_09,17466.0,-0.190554,0.133019,-0.417309,-0.270748,-0.230951,-0.163921,0.543062
ndmi_10,17466.0,-0.203787,0.131029,-0.416087,-0.288263,-0.235347,-0.168981,0.601874
ndvi_04,17466.0,0.512089,0.244112,-0.042130,0.281894,0.473881,0.757627,0.929851


## Ținta: anomalia de productivitate 2024
Principal: CLMS HR-VPP (productivitatea sezonieră, sezonul 1), descărcat din WEkEO în `data/raw/hrvpp/`.
Rezervă (până ajunge HR-VPP): anomalia NDVI iulie–august din Sentinel-2. Ambele sunt standardizate pe 2017–2023.

In [7]:
b = F[[f'ndvi_ja_{y}' for y in BASE_YEARS]]
F['ndvi_ja_z_2024'] = (F[f'ndvi_ja_{YEAR}'] - b.mean(axis=1)) / b.std(axis=1).replace(0, np.nan)

In [8]:
# HR-VPP: verifică întâi numele fișierelor descărcate și ajustează tiparul (sezonul 1, productivitate totală)
import glob, rasterio
from rasterio.merge import merge
from rasterstats import zonal_stats
HRVPP_DIR = RAW / 'hrvpp'
PARAM, SEASON = 'TPROD', 's1'      # <- ajustează după numele reale ale fișierelor
print(sorted(p.name for p in HRVPP_DIR.glob('*.tif'))[:10])

def hrvpp_files(year):
    return sorted(str(p) for p in HRVPP_DIR.glob('*.tif')
                  if str(year) in p.name and PARAM in p.name.upper() and SEASON in p.name.lower())

def parcel_mean(paths):
    """Media pe parcelă pentru un an. Tile-urile se grupează pe CRS (UTM 34 și 35 nu se pot lipi direct);
    pentru fiecare parcelă se păstrează media din zona UTM cu cei mai mulți pixeli valizi."""
    by_crs = {}
    for p in paths:
        with rasterio.open(p) as s:
            by_crs.setdefault(s.crs.to_string(), []).append(p)
    best, best_n = np.full(len(parcels), np.nan), np.zeros(len(parcels))
    for crs, group in by_crs.items():
        srcs = [rasterio.open(p) for p in group]
        try:
            geoms = parcels.to_crs(crs).geometry
            arr, tr = merge(srcs, bounds=tuple(geoms.total_bounds))   # doar fereastra parcelelor, nu tile-uri întregi
            st = zonal_stats(geoms, arr[0], affine=tr, nodata=srcs[0].nodata, stats=['mean', 'count'])
        finally:
            for s in srcs:
                s.close()
        m = np.array([np.nan if x['mean'] is None else x['mean'] for x in st], dtype=float)
        n = np.array([x['count'] for x in st], dtype=float)
        better = n > best_n
        best[better], best_n[better] = m[better], n[better]
    return best.tolist()

hr = {}
for y in BASE_YEARS + [YEAR]:
    paths = hrvpp_files(y)
    if paths:
        hr[y] = parcel_mean(paths)
if len(hr) == len(BASE_YEARS) + 1:
    H = pd.DataFrame(hr, index=parcels.parcel_id)
    z = (H[YEAR] - H[BASE_YEARS].mean(axis=1)) / H[BASE_YEARS].std(axis=1).replace(0, np.nan)
    F = F.merge(z.rename('tprod_z_2024'), left_on='parcel_id', right_index=True, how='left')
    F = F.merge(H.rename(columns=lambda y: f'tprod_{y}'), left_on='parcel_id', right_index=True, how='left')   # valorile anuale, pentru 03e
    print('ținta HR-VPP adăugată')
else:
    print('HR-VPP incomplet (ani găsiți:', sorted(hr), ') -> modelul folosește ținta de rezervă ndvi_ja_z_2024')

[]
HR-VPP incomplet (ani găsiți: [] ) -> modelul folosește ținta de rezervă ndvi_ja_z_2024


## LUCAS 2018 în zonă (licență personală: nu publici punctele individuale)

In [9]:
# data/private nu intră în git: rulează celula pe kernelul local (PC) sau urcă arhiva manual pe runtime
import zipfile
for z in PRIVATE.glob('*.zip'):
    if not (PRIVATE / z.stem).exists():
        zipfile.ZipFile(z).extractall(PRIVATE / z.stem)
csvs = sorted(PRIVATE.rglob('*.csv'))
if not csvs:
    print('LUCAS nu e pe acest kernel -> sar peste; stratul de sol folosește SoilGrids')
else:
    luc = pd.read_csv(csvs[0], low_memory=False)
    print(csvs[0].name, luc.columns.tolist())
    LUCAS_COLS = dict(id='POINTID', lat='TH_LAT', lon='TH_LONG', oc='OC')   # <- verifică după lista de mai sus
    missing = [c for c in LUCAS_COLS.values() if c not in luc.columns]
    assert not missing, f'Coloane lipsă: {missing} — corectează LUCAS_COLS'
    L = gpd.GeoDataFrame(luc, geometry=gpd.points_from_xy(luc[LUCAS_COLS['lon']], luc[LUCAS_COLS['lat']]), crs=4326)
    L = L.cx[AOI_BBOX[0]:AOI_BBOX[2], AOI_BBOX[1]:AOI_BBOX[3]].to_crs(CRS)
    L.to_parquet(PRIVATE / 'lucas_aoi.parquet')
    print(len(L), 'puncte LUCAS în zonă')

LUCAS nu e pe acest kernel -> sar peste; stratul de sol folosește SoilGrids


In [10]:
F.to_parquet(OUT / 'parcel_features.parquet')
if os.path.exists('/content'):     # pe PC comiți normal, din git
    push([OUT / 'parcel_features.parquet'], 'atribute pe parcelă')
print(F.shape)

(17466, 58)